In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

d:\ProteinPrediction\MainFiles\GPU_VENV\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


Convert data to tensors 

In [4]:
import numpy as np
import torch

X_train_tensor = torch.from_numpy(np.load("../data/Saved_Windows_features/X_train.npy")).float()
y_train_tensor = torch.from_numpy(np.load("../data/Saved_Windows_features/y_train.npy")).float()

X_val_tensor = torch.from_numpy(np.load("../data/Saved_Windows_features/X_val.npy")).float()
y_val_tensor = torch.from_numpy(np.load("../data/Saved_Windows_features/y_val.npy")).float()

X_test_tensor = torch.from_numpy(np.load("../data/Saved_Windows_features/X_test.npy")).float()
y_test_tensor = torch.from_numpy(np.load("../data/Saved_Windows_features/y_test.npy")).float()

Dataset Class

In [5]:
class ProteinDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

Dataloaders ( Mini Batch Training )

In [6]:
batch_size = 4096

train_loader = DataLoader(
    ProteinDataset(X_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    ProteinDataset(X_val_tensor, y_val_tensor),
    batch_size=batch_size
)

test_loader = DataLoader(
    ProteinDataset(X_test_tensor, y_test_tensor),
    batch_size=batch_size
)

MLP Model 

In [7]:
class MLPRegressor(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(451, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 4)
        )

    def forward(self, x):
        return self.net(x)

Initialize the model 

In [8]:
model = MLPRegressor().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
epochs = 30

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)

        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {train_loss/len(train_loader):.4f}")

Epoch 1/30 Loss: 892904.7935
Epoch 2/30 Loss: 886744.7531
Epoch 3/30 Loss: 879627.9314
Epoch 4/30 Loss: 870770.8507
Epoch 5/30 Loss: 859312.1398
Epoch 6/30 Loss: 846224.9319
Epoch 7/30 Loss: 829982.2649
Epoch 8/30 Loss: 809399.5715
Epoch 9/30 Loss: 792744.0345
Epoch 10/30 Loss: 763007.1946
Epoch 11/30 Loss: 730433.2229
Epoch 12/30 Loss: 683867.7407
Epoch 13/30 Loss: 660980.8357
Epoch 14/30 Loss: 629835.4160
Epoch 15/30 Loss: 606914.9710
Epoch 16/30 Loss: 585872.9771
Epoch 17/30 Loss: 577764.5846
Epoch 18/30 Loss: 560843.2295
Epoch 19/30 Loss: 560812.6199
Epoch 20/30 Loss: 548159.8635
Epoch 21/30 Loss: 534265.0803
Epoch 22/30 Loss: 525202.6445
Epoch 23/30 Loss: 523395.0707
Epoch 24/30 Loss: 508058.6519
Epoch 25/30 Loss: 511369.1410
Epoch 26/30 Loss: 498334.9872
Epoch 27/30 Loss: 494567.9935
Epoch 28/30 Loss: 482388.3858
Epoch 29/30 Loss: 481587.2762
Epoch 30/30 Loss: 482320.2525


In [13]:
model.eval()

preds = []
targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        preds.append(outputs.cpu().numpy())
        targets.append(y_batch.numpy())

preds = np.vstack(preds)
targets = np.vstack(targets)

In [14]:
rmse = np.sqrt(mean_squared_error(targets, preds))
mae = mean_absolute_error(targets, preds)
r2 = r2_score(targets, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 695.4342168746085
MAE: 228.09283447265625
R2: 0.5485751628875732
